## 高级用法
- 1.name 常在多agent协作时使用,区分不同agent,会在AIMessage上显示
- 2.system_prompt agent系统提示词,不会出现在回复中,两种形式:str/SystemMessage()
- 3.response_format 分为AutoStrategy,ProviderStrategy,ToolStrategy
    - AutoStrategy自动选择调用ProviderStrategy还是ToolStrategy
    - ProviderStrategy依靠模型供应商api结构化输出,不是每个供应商都支持
    - ToolStrategy依靠工具调用结构化输出,几乎所有模型都支持工具调用,通用性强,推荐使用
        - scheme :pydantic,typeddict,json_schema,dataclass
        - handle_errors :默认是true,在ToolMessage的content中显示错误类型,False遇到错误直接终止程序,返回错误,str在ToolMessage的content中显示相应字符串提示LLM,callable自定义函数处理错误,常见错误类型为MultipleStructuredOutputsError,返回的工具调用请求数量大于1时,StructuredOutputValidationError输出结构化验证错误，当输出格式不符合结构化要求时
        - tool_message_content :替换工具content,精简回复
- 4.stream 流式输出,七种模式["values", "updates", "checkpoints","tasks", "debug", "messages", "custom"
    - values:消息列表新增消息执行后，都会输出完整的消息列表所有消息，适用于每一步都要获取完整状态、状态持久化场景
    - updates:默认,每个步骤执行后，只增量更新最新的单条消息，用于监控Agent 执行进度，如观察Agent决定调用工具、工具执行结果等步骤
    - messages流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现流式打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。
    -tasks了解即可,输出模式该模式会输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生命周期。
    -debug了解即可,与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期
    - checkpoints输出模式该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景
    - custom通过 get_stream_writer 在工具或节点内部自定义发送的数据,用于输出业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标

In [6]:
from typing import Union
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override=True)
model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)



class UserInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")
class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

dataAgent = create_agent(
    model = model,
    name="data_agent",
    system_prompt=SystemMessage(content="你是一个专业的数据处理助手"),
    #system_prompt="你是一个专业的数据处理助手",
    response_format=ToolStrategy(#以工具调用形式结构化输出
        schema=Union[UserInfo, EventInfo],# 定义输出格式,Union表示可以是UserInfo也可以是EventInfo但只能是其中之一
        tool_message_content="结构化输出成功",#替换工具content,精简回复
        handle_errors=True,# 默认是true,在ToolMessage的content中显示错误类型,False遇到错误直接终止程序,返回错误,str在ToolMessage的content中显示相应字符串提示LLM,callable自定义函数处理错误,常见错误类型为MultipleStructuredOutputsError,返回的工具调用请求数量大于1时,StructuredOutputValidationError输出结构化验证错误，当输出格式不符合结构化要求时

    )
)

#response = dataAgent.invoke({"messages": [HumanMessage("请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15")]})

chunk = dataAgent.stream(
    {"messages": [HumanMessage("请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15")]},
    mode="updates",
)

for chunk in chunk:
    rprint(chunk)


{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='9b2844d6-aca8-4041-903c-6abd02d53fc8'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 126,
                    'prompt_tokens': 439,
                    'total_tokens': 565,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 55
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'f9f53696-57a1-4a85-8425-50bd5c3a05c6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='data_agent',
            id='lc_run--01a00af2-2f86-7bd2-b66f-ef0509d57e8d-0',
            tool_calls=[
                {
                    'name': 'UserInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com', 'phone': ''},
                    'id': 'call_00_5YmpkjG1JcHv96BG7d1c5629',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventInfo',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_01_z5i90NFgSS2QWm4zkJIE0804',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 439,
                'output_tokens': 126,
                'total_tokens': 565,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (UserInfo, EventInfo) when 
only one is expected.\n Please fix your mistakes.',
            name='UserInfo',
            id='2bb6435a-d47a-4972-a2b9-f968196f402f',
            tool_call_id='call_00_5YmpkjG1JcHv96BG7d1c5629'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (UserInfo, EventInfo) when 
only one is expected.\n Please fix your mistakes.',
            name='EventInfo',
            id='6def5480-5dba-430b-8315-6925d35fc248',
            tool_call_id='call_01_z5i90NFgSS2QWm4zkJIE0804'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 73,
                    'prompt_tokens': 642,
                    'total_tokens': 715,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 640},
                    'prompt_cache_hit_tokens': 640,
                    'prompt_cache_miss_tokens': 2
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '001e2037-29f7-4a71-98e2-0e4d9b6a816e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='data_agent',
            id='lc_run--01a00af2-345d-7f11-8d43-ddc097988989-0',
            tool_calls=[
                {
                    'name': 'UserInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com', 'phone': ''},
                    'id': 'call_00_hoqYMuWAcBMYLFScczCs6547',
